# Chapter 01: Euclid's Geometry

    ## Source Span

    Source orientation: printed pages 6-37, PDF pages 24-55. The source PDF is scanned, so this notebook does not depend on copied page text. It uses the span to identify the chapter's concepts and then rebuilds the lesson with original explanations, executable constructions, and locally generated artifacts.

    ## Chapter Question

    What does Euclid's system achieve, and where does a diagram quietly overpromise?

    Euclid's postulates support powerful constructions, while the parallel postulate behaves differently from local ruler-and-compass permissions. The central habit in this course is to treat every geometric sentence as something that can be represented in more than one way: a diagram, a finite model, an algebraic rule, a metric computation, or a dependency graph. That habit is especially important for this book because the historical narrative is not just background. The history records which claims seemed visually obvious, which claims resisted proof, and which claims became visible only after mathematicians learned to separate axioms from models.

    A reader should not need the textbook open while using this notebook. The notebook therefore states the working definitions it needs, explains how each computation should be read, and ends with small sanity checks. The checks are not a replacement for proof; they are a way to keep the visualization honest. If a diagram claims an invariant, the code records a numerical or structural witness for that invariant. If a construction is meant to compare geometries, the artifact names the comparison instead of serving as decoration.

    ## Translation Guide

    - postulates become construction permissions.
- proof steps become dependency edges.
- diagram facts are tested by perturbing the drawing.

    These translations are deliberately modest. They do not try to mechanize the whole chapter. Instead, they identify the parts of the chapter where a reader benefits from touching the geometry: moving a point, inspecting a model, checking an angle, or following a proof dependency. In a foundations chapter, the visual object may be a graph of assumptions. In a hyperbolic chapter, it may be a disk model. In a history chapter, it may be a timeline whose edges mark hidden assumptions rather than a list of dates.

    ## Route Through The Notebook

    1. read the postulates as operations.
2. build a construction diagram.
3. separate local construction from global parallel behavior.

    The route moves from language to inspection. First we name the concepts, then we build one structural visual and one geometric visual, and finally we run a small applied lab. The lab is intentionally small enough to modify: change a point, a parameter, or a model assumption and rerun the cells. If the outcome changes in the expected way, the chapter's main distinction has become operational rather than merely verbal.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Euclidean-and-Non-Euclidean-Geometries/chapter-01-euclids-geometry/01-euclids-geometry.ipynb",
  "course_dir": "Euclidean-and-Non-Euclidean-Geometries",
  "course_title": "Euclidean and Non-Euclidean Geometries",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Euclidean-and-Non-Euclidean-Geometries/chapter-01-euclids-geometry/01-euclids-geometry.ipynb",
  "jupyterlite": true,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Euclidean-and-Non-Euclidean-Geometries/chapter-01-euclids-geometry/01-euclids-geometry.ipynb",
  "notebook_title": "Chapter 01: Euclid's Geometry",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/classic.txt",
  "runtime_profile": "classic"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from pathlib import Path
import sys

import numpy as np

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate BOOK_ROOT")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_json
from utils.axioms import AXIOM_FAMILIES, DEPENDENCIES, MODEL_DICTIONARY
from utils.chapter_visuals import chapter_sanity, save_lab, save_relationship_map, save_scene

TOPIC = 'chapter-01-euclids-geometry'
ARTIFACT_DIR = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Writing artifacts to {ARTIFACT_DIR.relative_to(BOOK_ROOT)}")


## Conceptual Core

The visual sequence for this unit is organized around: Postulates, Construction, Diagram risk, Parallel attempt, Axiom audit. The first artifact is a relationship map. It should be read as a proof or interpretation diagram: arrows mean that one idea gives usable structure to the next. The second artifact is geometric. It makes the chapter's main distinction visible as an object that can be inspected directly.

For this unit the key idea is: Euclid's postulates support powerful constructions, while the parallel postulate behaves differently from local ruler-and-compass permissions. That sentence has two sides. On the formal side, we need precise permissions: what can be constructed, copied, ordered, measured, or inferred. On the model side, we need examples that make the permissions believable without smuggling in facts from a stronger geometry. The danger is subtle. A Euclidean drawing can accidentally make a non-Euclidean theorem look false, while a hyperbolic model drawn inside the Euclidean plane can accidentally make a curved fact look like an ordinary flat one. The notebook keeps those layers separate.

A useful working rule is this: a diagram may suggest, a model may interpret, a computation may test, but only the stated assumptions license a theorem. The artifacts below are built under that rule. They are meant to be inspected with a question in mind: which part of the picture is a convention of drawing, which part is an invariant of the model, and which part is a theorem waiting for proof?

## Standalone Study Notes

The chapter can be studied as a sequence of contrasts. One contrast is between local construction and global structure. Local construction is what a ruler, compass, incidence rule, or congruence rule lets us do immediately. Global structure is what happens when those local permissions are repeated across the whole plane. The parallel problem is powerful precisely because it is global: it is not settled by staring at one short segment or one small triangle.

A second contrast is between syntax and semantics. Syntax is the formal statement of an axiom or theorem. Semantics is a model in which the statement is interpreted. A finite incidence model, a Euclidean coordinate plane, a Poincare disk, a Klein disk, a sphere with antipodal identification, or a small grid without limiting points can all serve as semantic laboratories. When a theorem survives translation into many models, we gain confidence that the proof used only the intended assumptions. When it fails in a model, the failure is not noise; it is information about which assumption was missing.

A third contrast is between measurement and proof. Measurement supplies evidence and intuition. Proof explains why the observation is forced by assumptions. This notebook keeps measurement visible because the subject is easy to misread without it. Angle sums, geodesic shapes, closure tables, and model maps are all chosen so that the reader can see what the formal language is trying to protect.


## Definition Bank And Equations

    - **Postulate.** a licensed construction or relation rather than a fact read directly from a drawing.
- **Diagram risk.** the danger that a special picture supplies an unstated assumption.
- **Construction dependency.** the ordered record of which objects were allowed before later objects were drawn.

    The definitions above are the local vocabulary for this notebook. They are intentionally stated in operational language because the goal is to connect the synthetic discussion to artifacts that can be inspected and rerun. A reader should be able to point to a plotted object, model table, or numerical check and say which definition it is representing.

    The equations or symbolic checks used in this unit are:

    - `$equilateral check: |AB|=|AC|=|BC|$`
- `$parallel claim: through P not on l, exactly one candidate is Euclidean$`

    These formulas are not a substitute for the synthetic proofs. They are a compact way to make the chapter's claims testable. When a formula appears again in a sanity check, it is being used as a consistency witness for the artifact, not as copied textbook prose.


## Worked Example

The equilateral-triangle construction is a clean example of local postulates at work. Given AB, draw the circle centered at A through B and the circle centered at B through A. Their intersection C gives AC=AB and BC=AB because both are radii. The proof does not need the parallel postulate. That contrast is the lesson: some Euclidean facts are local construction facts, while the parallel postulate controls a global behavior of the whole plane.

To read this as a proof exercise, separate the construction from the inference. The construction tells us which objects are present. The inference tells us which relation is licensed by the definitions or axioms. The computational version mirrors that separation: first the code creates a model or diagram, then the final check records the invariant that should survive rerunning the notebook.


In [ ]:
chapter_profile = {
    "title": "Euclid's Geometry",
    "source_printed_pages": '6-37',
    "source_pdf_pages": '24-55',
    "visual_center": 'postulates, construction diagrams, diagram ambiguity, parallel-postulate attempts',
    "chapter_question": "What does Euclid's system achieve, and where does a diagram quietly overpromise?",
    "translation": ['postulates become construction permissions', 'proof steps become dependency edges', 'diagram facts are tested by perturbing the drawing'],
    "route": ['read the postulates as operations', 'build a construction diagram', 'separate local construction from global parallel behavior'],
    "relationship_labels": ['Postulates', 'Construction', 'Diagram risk', 'Parallel attempt', 'Axiom audit'],
    "scene_kind": 'euclid-construction',
    "lab_kind": 'diagram-perturbation',
    "sanity_kind": 'euclid-construction',
    "artifact_names": {'relationship': 'euclid-postulate-dependency-map.png', 'scene': 'equilateral-circle-construction.png', 'lab': 'diagram-perturbation-risk-lab.png'},
}
chapter_profile


## Visual Storyboard

The storyboard chooses a compact set of generated artifacts for this chapter. The relationship map provides a view of proof or interpretation structure. The geometric scene gives an inspectable construction, model, or surface. The lab plot records a numerical or finite diagnostic. Each artifact filename names the concept it carries, and each artifact is regenerated from code so the reader can modify it.


In [ ]:
relationship_path = ARTIFACT_DIR / chapter_profile["artifact_names"]["relationship"]
save_relationship_map(chapter_profile["relationship_labels"], relationship_path, chapter_profile["title"] + ": structure")
display_artifact(relationship_path)


In [ ]:
geometry_path = ARTIFACT_DIR / chapter_profile["artifact_names"]["scene"]
save_scene(chapter_profile["scene_kind"], geometry_path)
display_artifact(geometry_path)


## Applied Lab

The lab turns the chapter's main contrast into a small experiment. Treat the plotted quantity as a diagnostic rather than as a final theorem. The point is to learn what would change if the underlying geometry or axiom choice changed. For example, a parallel-count plot separates Euclidean, hyperbolic, and elliptic behavior; a closure table checks whether transformations stay inside a proposed symmetry set; a defect plot translates the statement "angle sum is less than two right angles" into a measurable area rule.

When adapting the lab, change one assumption at a time. In synthetic geometry, changing two assumptions at once often hides the reason a conclusion failed. In computational work the same discipline appears as a small parameter sweep or a finite model table. The table or curve is not the proof, but it is a compact way to see where the proof should look.


In [ ]:
lab_path = ARTIFACT_DIR / chapter_profile["artifact_names"]["lab"]
save_lab(chapter_profile["lab_kind"], lab_path)
display_artifact(lab_path)


## Sanity Checks

The checks below make the notebook auditable. They assert that the visual artifacts exist, record a few small invariants, and save a JSON summary next to the images. These values are intentionally simple: they are there to catch stale paths, blank artifacts, and broken helper functions before a reader encounters them.


In [ ]:
final_sanity = {
    "topic": TOPIC,
    "artifact_count": 3,
    "axiom_family_count": len(AXIOM_FAMILIES),
    "model_dictionary_count": len(MODEL_DICTIONARY),
    "dependency_count": len(DEPENDENCIES),
}
final_sanity.update(chapter_sanity(chapter_profile["sanity_kind"]))
sanity_path = save_json(final_sanity, ARTIFACT_DIR / "final-sanity.json")
assert final_sanity["sanity_kind"] == chapter_profile["sanity_kind"]
assert_artifacts([relationship_path, geometry_path, lab_path, sanity_path], min_bytes=64)
final_sanity


## Takeaways

- The main standalone lesson is: Euclid's postulates support powerful constructions, while the parallel postulate behaves differently from local ruler-and-compass permissions.
- The structural artifact records the dependency or interpretation pattern; the geometric artifact records what the pattern looks like in a concrete model.
- The applied lab is a reusable check: it lets a reader perturb a parameter and watch which conclusion is stable.
- The source span supplies orientation, but the course notebook supplies its own definitions, examples, visuals, and sanity checks.

A good next reading move is to reopen the earlier notebooks whenever a later model seems visually surprising. Most surprises in non-Euclidean geometry are not caused by a single strange diagram; they come from moving a familiar proof step into a world where one hidden assumption has been removed or replaced.
